![](../images/img1.png)
![](../images/img2.png)

# Steps for Simple Chat App  

![image.png](../images/img3.png)
![image.png](../images/img4.png)
![image.png](../images/img5.png)
![image.png](../images/img6.png)
![image.png](../images/img7.png)

# Getting an API Key

## Step One: Navigate to the Anthropic API Console

## Step Two: Click the "Get API Keys" button

## Step Three: Click the "Create Key" button

## Step Four: Enter a workspace and name for your key

## Step Five: Copy the key

In [13]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv(override=True) 
API_KEY = os.getenv("GEMINI_API_KEY") 

client  = genai.Client(api_key=API_KEY)


In [14]:
response = client.models.generate_content(
    model="gemini-2.5-flash", 
    contents="Are you up? Yes or No.",
    config={"max_output_tokens":100}
)
print(response.text)   


Yes


![](../images/img8.png)

In [15]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents ="What is Javascript?Give me a one liner",
    config={"max_output_tokens":1000}
)
print(response.text)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Give me another line",
    config={"max_output_tokens":1000}
)

print(response.text)

JavaScript is a versatile programming language that makes web pages interactive and is widely used for web, mobile, and desktop application development.
Sure, I can give you another line!

But to make sure it's helpful, could you tell me what kind of line you're looking for?

*   A line for a poem or song?
*   A line of dialogue for a story?
*   A line of code?
*   A punchline for a joke?
*   Just a random thought?

Let me know, and I'll generate one!


This shows that conversation history is not stored.

Each request you make is completely independent, with no memory of previous exchanges.

This means that if you want to have a multi-turn conversation where Claude remembers context from earlier messages, you need to handle the conversation state yourself.

---

# Maintaining Conversation Context

To maintain conversation context, you need to do two things:

1. Manually maintain a list of all messages in your code
2. Send the complete message history with every request

---

# Conversation Flow

Here is the flow that actually works:

1. Send your initial user message to Claude
2. Take Claude's response and add it to your message list as an assistant message
3. Add your follow-up question as another user message
4. Send the entire conversation history to Claude

This way, the model receives the previous context every time and can generate context-aware responses.

In [16]:
def add_user_messages(messages,text):
    messages.append({"role":"user", "parts":[{"text":text}]})


def add_model_message(messages,text):
    messages.append({"role":"user", "parts":[{"text":text}]})

def chat(messages):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages,
        config={"max_output_tokens":1000}
    )
    return response.text


In [17]:
messages_js = []

add_user_messages(messages_js, text="What is Javascript?Give a one liner.")
reply = chat(messages_js)
add_model_message(messages_js, reply)
print(reply)

add_user_messages(messages_js, text="Give me another line")
reply = chat(messages_js)
add_model_message(messages_js, reply)
print(reply)

add_user_messages(messages_js, text="What were the two line that you gave me")
reply = chat(messages_js)
add_model_message(messages_js, reply)
print(reply)


JavaScript is a versatile programming language that brings interactivity and dynamic behavior to websites, and is also widely used for server-side development and other applications.
JavaScript is a powerful programming language that adds interactivity to web pages and is used for a wide range of other applications.

It's the standard for client-side web development, making
Here are the two lines I gave you:

1.  JavaScript is a versatile programming language that brings interactivity and dynamic behavior to websites, and is also widely used for server-side development and other applications.
2.  JavaScript is a powerful programming language that adds interactivity to web pages and is used for a wide range of other applications.


![](../images/img9.png)

In [21]:
del input

In [25]:
messages = []

while True:
    user_input = input("Enter your prompt")

    if user_input.lower() in ["exit","quit","end","bye"]:
        print("Ending conversation Goodbye!")
        break 

    add_user_messages(messages, user_input)
    reply = chat(messages)
    add_model_message(messages, reply)
    print(reply)

2
10
Ending conversation Goodbye!


# System Prompts

System prompts are a powerful way to customize how Claude responds to user input.

Instead of getting generic answers, you can shape Claude’s:

- Tone
- Style
- Behavior
- Response approach

to match your specific use case.

---

# Example: Math Tutor Chatbot

Consider building a math tutor chatbot.

When a student asks:

> "How do I solve \(5x + 2 = 3\) for \(x\)?"

you want Claude to behave like a real tutor instead of immediately giving the answer.

---

# A Good Math Tutor Should

- Initially give hints rather than complete solutions
- Patiently guide students step by step
- Explain the reasoning behind each step
- Show similar example problems
- Encourage students to think independently

---

# You Do NOT Want Claude To

- Immediately provide direct answers
- Solve everything without explanation
- Tell students to "just use a calculator"
- Skip intermediate reasoning steps

---

# Why System Prompts Matter

System prompts help define the AI's role and behavior before the conversation starts.

For example, a system prompt for a tutor might instruct Claude to:

- Act like a supportive teacher
- Avoid directly revealing answers
- Prioritize learning over speed
- Give hints first
- Explain concepts clearly

This allows developers to create specialized AI assistants tailored for education, customer support, coding help, healthcare guidance, and more.

![](../images/img10.png)

In [34]:
def chat(messages, system_prompt = None):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents= messages,
        config = genai.types.GenerateContentConfig(
            system_instruction= system_prompt,
            max_output_tokens= 1000,
        )
    )
    return response.text

In [36]:
messages = []

system_prompt = "You are a helpful math tutor. Only answer math-related questions. Keep answers short and simple."

while True:
    user_input = input("Enter your prompt")

    if user_input.lower() in ["exit","quit","end","bye"]:
        print("Ending conversation Goodbye!")
        break 

    add_user_messages(messages, user_input)
    reply = chat(messages)
    add_model_message(messages, reply)
    print(reply)

2
Here are the answers to your questions:

1.  **2**

2.  Here's a recipe for classic Chicken Momo:

    **Chicken Momo Recipe**

    This recipe makes approximately 30-40 momos.

    **I. Ingredients**

    **For the Dough:**
    *   2 cups all-purpose flour (maida)
    *   1/2 teaspoon salt
    *   1/2 cup + 2-3 tablespoons water (or as needed)
    *   1 teaspoon oil (optional, for softness)

    **For the Filling:**
    *   1 lb (approx. 450g) ground chicken (preferably thigh meat for juiciness, or a mix)
    *   1 medium onion, very finely chopped
    *   1/4 cup spring onion greens, finely chopped (optional, for garnish and flavor)
    *   1 tablespoon ginger, minced
    *   1 tablespoon garlic, minced
    *   1-2 green chilies, minced (adjust to taste)
    *   1/4 cup fresh cilantro, chopped
    *   1 tablespoon soy sauce (light)
    *   1 teaspoon black pepper powder
    *   1/2 teaspoon garam masala (optional, for extra warmth)
    *   1 tablespoon sesame oil (optional, for aut

In [37]:
messages = []

system_prompt = "You are a helpful math tutor. Only answer math-related questions. Keep answers short and simple."

while True:
    user_input = input("Enter your prompt")

    if user_input.lower() in ["exit","quit","end","bye"]:
        print("Ending conversation Goodbye!")
        break 

    add_user_messages(messages, user_input)
    reply = chat(messages, system_prompt)
    add_model_message(messages, reply)
    print(reply)

2
1 + 1 = 2

I am a math tutor and can only answer math-related questions.
Ending conversation Goodbye!


In [39]:
#Practice

def chat(messages, system_prompt = None):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents= messages,
        config = genai.types.GenerateContentConfig(
            system_instruction= system_prompt,
            max_output_tokens= 10000,
        )
    )
    return response.text

messages = []
add_user_messages(messages, "Write a python function that checks a string for duplicate characters")
answer = chat(messages)
print(answer)

There are several ways to check a string for duplicate characters in Python. Here are a few common approaches, from a simple and efficient one using a set to others.

---

### Method 1: Using a Set (Most Pythonic and Efficient)

This method iterates through the string, adding each character to a `set`. A set only stores unique elements. If you try to add a character that's already in the set, it means you've found a duplicate.

```python
def has_duplicate_chars_set(s):
    """
    Checks if a string contains duplicate characters using a set.

    Args:
        s (str): The input string to check.

    Returns:
        bool: True if duplicate characters are found, False otherwise.
    """
    seen_characters = set()
    for char in s:
        if char in seen_characters:
            return True  # Found a duplicate!
        seen_characters.add(char)
    return False  # No duplicates found after checking all characters

# --- Test Cases ---
print("--- Set Method ---")
print(f"'hello': {has

In [40]:
messages = []
add_user_messages(messages, "Write a python function that checks a string for duplicate characters")
answer = chat(messages, system_prompt="You are a python developer who writes very concise code")
print(answer)

```python
def has_duplicates(s: str) -> bool:
    """
    Checks if a string contains any duplicate characters.
    """
    return len(s) != len(set(s))
```


In [ ]:
messages = []

while True:
    user_input = input("Enter your prompt")

    if user_input.lower() in ["exit","quit","end","bye"]:
        print("Ending conversation Goodbye!")
        break 

    add_user_messages(messages, user_input)
    reply = chat(messages)
    add_model_message(messages, reply)
    print(reply)

2
10
Ending conversation Goodbye!


# How Claude Generates Text

Before diving into **temperature**, it helps to understand Claude’s text generation process.

When you send Claude a prompt like:

> "What do you think?"

it goes through three key steps:

---

## 1. Tokenization
Breaking your input into smaller chunks called **tokens**.

These tokens can be words, subwords, or characters depending on the context.

---

## 2. Prediction
Claude calculates probabilities for all possible next tokens based on the input context.

It essentially asks:

> "What is the most likely next word?"

---

## 3. Sampling
A token is selected based on the probability distribution.

This step determines how deterministic or creative the output will be.

---

![](../images/img11.png)
![](../images/img12.png)

---

# Choosing the Right Temperature

**Temperature** controls randomness in Claude’s responses.

- Low temperature → more deterministic and factual
- High temperature → more creative and diverse

---

## Low Temperature (0.0 – 0.3)

Best for:

- Factual responses
- Coding assistance
- Data extraction
- Content moderation

---

## Medium Temperature (0.4 – 0.7)

Best for:

- Summarization
- Educational content
- Problem-solving
- Creative writing with constraints

---

## High Temperature (0.8 – 1.0)

Best for:

- Brainstorming
- Creative writing
- Marketing content
- Joke generation

In [48]:
def chat(messages, system_prompt = None, temperature = 1.0):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents= messages,
        config = genai.types.GenerateContentConfig(
            system_instruction= system_prompt,
            max_output_tokens= 10000,
            temperature=temperature
        )
    )
    return response.text

In [54]:
messages = []

add_user_messages(messages , "Generate a one sentence movie idea. Only one movie idea.")
answer = chat(messages = messages,temperature= 0.0)
print(answer)

A chef discovers his food can transport people to their happiest memories, but each dish requires him to sacrifice one of his own.


In [55]:
messages = []

add_user_messages(messages ,  "Generate a one sentence movie idea")
answer = chat(messages = messages,temperature= 1.0)
print(answer)

In a future where social media likes literally determine your lifespan, a desperate influencer must commit increasingly outlandish acts to simply survive another day.


# Response Streaming

When building chat applications with Claude, there is a major user experience challenge:

Responses can take **10–30 seconds** to generate, leaving users staring at a loading spinner.

---

## Solution: Response Streaming

**Response streaming** allows users to see the answer appear gradually, chunk by chunk, as Claude generates it.

This makes the application feel much more:

- Responsive
- Interactive
- Real-time

---

## How Streaming Works

With streaming enabled:

1. Claude immediately sends an initial response confirming the request is received.
2. Claude begins generating the output.
3. You receive a continuous sequence of small **events**.
4. Each event contains a small piece (token/chunk) of the final response.

---

## Result

Instead of waiting for the full response, the user sees text appearing progressively, like real-time typing.

---

![](../images/img12.png)

In [62]:
messages = []

add_user_messages(messages, "Give me a one liner on a fake animal")

response = client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents=messages,
    config={"max_output_tokens": 1000}
)

full_reply = ""

for chunk in response:
    if chunk.text is not None:       
        print(chunk.text, end="", flush=True)
        full_reply += chunk.text

print()
add_model_message(messages, full_reply)

The **Grumblepuff** eats bad moods for breakfast, leaving behind only giggles.


# Generating Structured Data with Claude

When you need Claude to generate structured data like:

- JSON
- Python code
- Bulleted lists

you’ll often run into a common problem: Claude tries to be helpful by adding explanatory text around the output.

While this is usually useful, it becomes an issue when you only need **raw data with no extra text**.

In [72]:

load_dotenv(override=True) 
API_KEY = os.getenv("GEMINI_API_KEY") 

client  = genai.Client(api_key=API_KEY)


In [73]:
messages = []
add_user_messages(messages , text="Give me information about Bengal Tiger as JSON. Keep JSON size small")

response  = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config = genai.types.GenerateContentConfig(
        max_output_tokens = 10000,
    )
)


print(response.text)

```json
{
  "name": "Bengal Tiger",
  "sci_name": "Panthera tigris tigris",
  "status": "Endangered",
  "habitat": "Grasslands, forests, mangroves",
  "diet": "Carnivore",
  "range": ["India", "Bangladesh", "Nepal", "Bhutan"],
  "avg_weight_kg": {
    "male": "180-250",
    "female": "100-160"
  }
}
```


In [74]:
messages = []
schema = {
    "type": "object",
    "properties": {
        "name":        {"type": "string"},
        "habitat":     {"type": "string"},
        "fun_fact":    {"type": "string"},
        "endangered":  {"type": "boolean"}
    }
}

add_user_messages(messages , text="Give me information about Bengal Tiger as JSON. Keep JSON size small")

response  = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config = genai.types.GenerateContentConfig(
        max_output_tokens = 10000,
        response_mime_type="application/json",  
        response_schema=schema 
    )
)


print(response.text)

{"name":"Bengal Tiger","habitat":"Indian subcontinent","fun_fact":"Largest cat species","endangered":true}
